In [9]:
import os
import sys
import time
import io
import zipfile
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

# 프로젝트 루트 경로 설정 (settings.py가 있는 위치를 기준으로)
# 현재 노트북 위치: MyEggBasket-AI/collectingDisclosure.ipynb
# 프로젝트 루트: MyEggBasket-AI/
current_dir = Path(os.getcwd())
project_root = current_dir  # 노트북이 루트 바로 아래 폴더나 루트에 있다면 조정 필요
sys.path.append(str(project_root))

try:
    from ai_pipeline.config import settings
    DART_API_KEY = os.getenv("DART_API_KEY")
    
except ImportError:
    # settings.py를 못 찾거나 로컬 테스트용
    from dotenv import load_dotenv
    env_path = Path(os.getcwd()) / ".env"
    load_dotenv(dotenv_path=env_path)
    DART_API_KEY = os.getenv("DART_API_KEY")

if not DART_API_KEY:
    raise ValueError("⚠ DART_API_KEY가 환경변수에 설정되지 않았습니다.")

print(f"✅ API Key Loaded: {DART_API_KEY[:4]}...****")

✅ API Key Loaded: 1a2d...****


In [12]:
def get_corp_code_list(api_key):
    """
    OpenDART 고유번호 API를 호출하여 전체 기업 목록을 DataFrame으로 반환합니다.
    URL: https://opendart.fss.or.kr/api/corpCode.xml
    """
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    params = {'crtfc_key': api_key}

    print("⏳ 고유번호(corp_code) 다운로드 및 파싱 중...")

    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()

        # Zip 파일 처리
        with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
            # zip 안에 있는 XML 파일명 찾기 (보통 CORPCODE.xml)
            xml_filename = z.namelist()[0]
            with z.open(xml_filename) as f:
                tree = ET.parse(f)
                root = tree.getroot()

        # XML 파싱
        data_list = []
        for child in root.findall('list'):
            corp_code = child.find('corp_code').text
            corp_name = child.find('corp_name').text.strip()
            stock_code = child.find('stock_code').text.strip()
            modify_date = child.find('modify_date').text

            data_list.append({
                'corp_code': corp_code,
                'corp_name': corp_name,
                'stock_code': stock_code,
                'modify_date': modify_date
            })

        df = pd.DataFrame(data_list)
        print(f"✅ 전체 기업 목록 수집 완료: {len(df)}개")
        return df

    except Exception as e:
        print(f"❌ Error fetching corp_code: {e}")
        return None

# 실행
corp_code_df = get_corp_code_list(DART_API_KEY)
# 상위 5개 확인
display(corp_code_df.head())

⏳ 고유번호(corp_code) 다운로드 및 파싱 중...
✅ 전체 기업 목록 수집 완료: 114720개


,corp_code,corp_name,stock_code,modify_date
0,00434003,다코,,20170630
1,00430964,굿앤엘에스,,20170630
2,00388953,크레디피아제이십오차유동화전문회사,,20170630
3,00179984,연방건설산업,,20170630
4,00420143,브룩스피알아이오토메이션잉크,,20170630


In [13]:
def get_disclosure_list(api_key, bgn_de, end_de, corp_code=None, pblntf_ty=None, last_reprt_at='N'):
    """
    OpenDART 공시검색 API 호출
    URL: https://opendart.fss.or.kr/api/list.json

    Params:
        corp_code: 고유번호 (없으면 전체 기업 검색)
        bgn_de: 시작일 (YYYYMMDD)
        end_de: 종료일 (YYYYMMDD)
        pblntf_ty: 공시유형 (A:정기공시, B:주요사항, I:거래소공시 등) - 없으면 전체
    """
    url = "https://opendart.fss.or.kr/api/list.json"
    results = []
    page_no = 1
    page_count = 100 # 최대 100건씩 요청

    while True:
        params = {
            'crtfc_key': api_key,
            'bgn_de': bgn_de,
            'end_de': end_de,
            'page_no': page_no,
            'page_count': page_count,
            'last_reprt_at': last_reprt_at
        }

        if corp_code:
            params['corp_code'] = corp_code
        if pblntf_ty:
            params['pblntf_ty'] = pblntf_ty

        try:
            resp = requests.get(url, params=params)
            data = resp.json()

            status = data.get('status')
            if status != '000':
                # 013: 조회된 데이터 없음
                if status == '013':
                    break
                print(f"⚠ API Warning/Error: {data.get('message')} (Code: {status})")
                break

            items = data.get('list', [])
            results.extend(items)

            # 페이지네이션 처리
            total_page = int(data.get('total_page', 1))
            if page_no >= total_page:
                break

            page_no += 1
            time.sleep(0.1) # API 부하 방지용 딜레이

        except Exception as e:
            print(f"❌ Error fetching disclosure list: {e}")
            break

    return pd.DataFrame(results)

In [14]:
from datetime import datetime, timedelta

# 예시: 최근 7일간의 모든 공시 수집
end_date = datetime.now()
start_date = end_date - timedelta(days=7)

str_start = start_date.strftime("%Y%m%d")
str_end = end_date.strftime("%Y%m%d")

print(f"🔍 공시 수집 기간: {str_start} ~ {str_end}")

# corp_code 없이 호출하면 해당 기간의 '모든 기업' 공시를 가져옵니다.
# 이것이 기업별로 루프를 도는 것보다 API 호출 수를 수천 배 아낄 수 있습니다.
all_disclosure_df = get_disclosure_list(
    DART_API_KEY,
    bgn_de=str_start,
    end_de=str_end,
    # pblntf_ty='B' # 필요한 경우 B:주요사항보고 등으로 필터링 가능
)

if not all_disclosure_df.empty:
    print(f"✅ 총 {len(all_disclosure_df)}건의 공시 수집 완료")

    # [데이터 병합] 수집된 공시 정보에는 종목코드(stock_code)가 없을 수 있으므로
    # Step 2에서 구한 corp_code_df와 병합하여 정보를 보강합니다.
    # corp_code를 기준으로 병합
    final_df = pd.merge(
        all_disclosure_df,
        corp_code_df[['corp_code', 'stock_code', 'modify_date']],
        on='corp_code',
        how='left'
    )

    # 필요한 컬럼 정리
    final_df = final_df[['rcept_dt', 'corp_cls', 'corp_name', 'stock_code_y', 'report_nm', 'rcept_no', 'flr_nm']]
    final_df.rename(columns={'stock_code_y': 'stock_code'}, inplace=True)

    display(final_df.head())

    # CSV 저장
    final_df.to_csv("recent_disclosures.csv", index=False, encoding="utf-8-sig")
else:
    print("해당 기간에 조회된 공시가 없습니다.")

🔍 공시 수집 기간: 20251126 ~ 20251203
✅ 총 6411건의 공시 수집 완료


,rcept_dt,corp_cls,corp_name,stock_code,report_nm,rcept_no,flr_nm
0,20251203,Y,롯데에너지머티리얼즈,020150,최대주주등소유주식변동신고서,20251203800341,롯데에너지머티리얼즈
1,20251203,Y,대웅제약,069620,최대주주등소유주식변동신고서,20251203800339,대웅제약
2,20251203,K,넥사다이내믹스,351320,주식등의대량보유상황보고서(약식),20251203000227,와이에이치투자조합 1호
3,20251203,Y,종근당,185750,임원ㆍ주요주주특정증권등소유상황보고서,20251203000225,종근당홀딩스
4,20251203,E,신보2022제15차유동화전문유한회사,,자산유동화관련중요사항발생등보고서(기타),20251203000224,신보2022제15차유동화전문유한회사


In [15]:
import os
import time
import re
import requests
import pandas as pd
from io import BytesIO
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

# ==========================================
# 1. 설정 및 API 키
# ==========================================
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DART_API_KEY = os.getenv("DART_API_KEY")
if not DART_API_KEY:
    # 테스트용 키가 없다면 에러 발생
    raise ValueError("⚠ DART_API_KEY가 환경변수에 설정되지 않았습니다.")

print(f"✅ API Key Loaded: {DART_API_KEY[:4]}...****")

# ==========================================
# 2. OpenDART API 통합 수집 클래스
# ==========================================
class OpenDartIntegratedCollector:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://opendart.fss.or.kr/api"

    def _request(self, endpoint, params):
        url = f"{self.base_url}/{endpoint}"
        params['crtfc_key'] = self.api_key
        try:
            resp = requests.get(url, params=params)
            data = resp.json()
            if data.get('status') == '000':
                return data.get('list', [])
            return None
        except Exception as e:
            return None

    def get_corp_code(self):
        """고유번호 가져오기"""
        url = f"{self.base_url}/corpCode.xml"
        params = {'crtfc_key': self.api_key}
        try:
            resp = requests.get(url, params=params)
            with zipfile.ZipFile(BytesIO(resp.content)) as z:
                with z.open(z.namelist()[0]) as f:
                    tree = ET.parse(f)
                    root = tree.getroot()
            data = []
            for child in root.findall('list'):
                data.append({
                    'corp_code': child.find('corp_code').text,
                    'stock_code': child.find('stock_code').text.strip(),
                })
            return pd.DataFrame(data)
        except:
            return pd.DataFrame()

    def get_disclosure_list(self, bgn_de, end_de, corp_cls='Y'):
        """공시 검색 (정기보고서만)"""
        endpoint = "list.json"
        results = []
        page_no = 1

        while True:
            params = {
                'bgn_de': bgn_de, 'end_de': end_de, 'corp_cls': corp_cls,
                'page_no': page_no, 'page_count': 100, 'pblntf_ty': 'A' # 정기공시
            }
            data = self._request(endpoint, params)
            if not data: break
            results.extend(data)
            if len(data) < 100: break
            page_no += 1
            time.sleep(0.1)
        return results

    # ---------------------------------------------------------
    # 상세 데이터 처리 로직 (One Row로 만들기 위한 요약 처리)
    # ---------------------------------------------------------

    def process_finance(self, data):
        """재무제표에서 핵심 계정만 추출"""
        if not data: return {}

        # 연결(CFS) 우선, 없으면 별도(OFS)
        df = pd.DataFrame(data)
        target_df = df[df['fs_div'] == 'CFS']
        if target_df.empty:
            target_df = df[df['fs_div'] == 'OFS']

        result = {}
        # 계정명 매핑 (정규식 활용)
        mappings = {
            '매출액': 'fin_revenue',
            '수익(매출액)': 'fin_revenue',
            '영업이익': 'fin_op_income',
            '영업이익(손실)': 'fin_op_income',
            '당기순이익': 'fin_net_income',
            '당기순이익(손실)': 'fin_net_income',
            '자산총계': 'fin_total_assets',
            '부채총계': 'fin_total_liabilities',
            '자본총계': 'fin_total_equity'
        }

        for _, row in target_df.iterrows():
            acct = row['account_nm'].replace(" ", "") # 공백제거 비교
            amt = row['thstrm_amount']

            # 금액 숫자 변환
            try:
                val = float(amt.replace(',', ''))
            except:
                val = 0

            # 매핑 확인
            for key, col_name in mappings.items():
                if key in acct:
                    result[col_name] = val

        return result

    def process_dividend(self, data):
        """배당금 (보통주 현금배당 기준)"""
        if not data: return {}
        for item in data:
            if item.get('se') == '주당 현금배당금(원)' and item.get('stock_knd') == '보통주':
                return {'div_dps_common': item.get('thstrm', 0)}
        return {}

    def process_major_shareholder(self, data):
        """최대주주 (가장 지분율 높은 사람)"""
        if not data: return {}
        try:
            top = data[0]
            return {
                'div_dps_common': top.get('div_dps_common'),
                # major_shareholder_name 및 major_shareholder_ratio 모두 더 이상 수집하지 않음
            }
        except:
            return {}

    def process_employees(self, data):
        """직원 현황 (합계)"""
        if not data: return {}
        total_emp = 0
        avg_salary = 0
        count = 0

        for item in data:
            # 합계 행을 찾거나 다 더함
            if '합계' in item.get('se', '') or '남여 공통' in item.get('se', ''):
                 return {
                     'emp_total_count': item.get('sm'),
                     'emp_avg_salary': item.get('jan_avrg_salary')
                 }
        # 합계 행이 없으면 첫번째 행이라도 가져옴
        if data:
            return {'emp_total_count': data[0].get('sm')}
        return {}

    def process_auditor(self, data):
        """감사의견"""
        if not data: return {}
        # 최근 사업연도 의견
        return {
            'auditor_name': data[0].get('audit_nm'),
            'audit_opinion': data[0].get('audit_opnn')
        }

    def process_simple_sum(self, data, field_name, out_col):
        """단순 잔액 합계 (미상환 잔액 등)"""
        if not data: return {}
        total = 0.0
        for item in data:
            val_str = item.get(field_name, '0')
            try:
                total += float(val_str.replace(',', ''))
            except:
                pass
        return {out_col: total} if total > 0 else {}


# ==========================================
# 3. 메인 실행 및 데이터 병합
# ==========================================
def run_integrated_collection(limit_companies=5):
    collector = OpenDartIntegratedCollector(DART_API_KEY)

    # 1. 종목코드 매핑 준비
    corp_df = collector.get_corp_code()

    # 2. 공시 목록 조회 (최근 3개월)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)

    print("🔍 공시 목록 조회 중...")
    reports_kospi = collector.get_disclosure_list(start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d"), 'Y')
    reports_kosdaq = collector.get_disclosure_list(start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d"), 'K')

    all_reports = reports_kospi + reports_kosdaq
    print(f"✅ 총 {len(all_reports)}건의 정기보고서 발견.")

    final_data_list = []

    # 3. 상세 데이터 수집 (Limit 적용)
    count = 0
    for report in all_reports:
        if count >= limit_companies:
            print(f"🛑 테스트 제한({limit_companies}개) 도달.")
            break

        # 보고서 정보 파싱
        report_nm = report['report_nm']
        year_match = re.search(r'\((\d{4})', report_nm)
        if not year_match: continue

        bsns_year = year_match.group(1)
        # 보고서 코드 결정
        if ".12)" in report_nm or "사업보고서" in report_nm: reprt_code = "11011"
        elif ".03)" in report_nm or "1분기" in report_nm: reprt_code = "11013"
        elif ".06)" in report_nm or "반기" in report_nm: reprt_code = "11012"
        elif ".09)" in report_nm or "3분기" in report_nm: reprt_code = "11014"
        else: continue

        print(f"[{count+1}] {report['corp_name']} ({report_nm}) 데이터 수집 중...")

        # 기본 행 데이터 구성
        row_data = {
            'corp_code': report['corp_code'],
            'corp_name': report['corp_name'],
            'report_name': report_nm,
            'rcept_no': report['rcept_no'],
            'rcept_dt': report['rcept_dt'],
            'bsns_year': bsns_year,
            'reprt_code': reprt_code
        }

        params = {'corp_code': report['corp_code'], 'bsns_year': bsns_year, 'reprt_code': reprt_code}

        # [API 1] 재무제표 (가장 중요)
        fin_data = collector._request("fnlttSinglAcnt.json", params.copy())
        row_data.update(collector.process_finance(fin_data))

        # [API 2] 배당 (사업보고서인 경우만 주로 존재)
        if reprt_code == '11011':
            div_data = collector._request("alotMatter.json", params.copy())
            row_data.update(collector.process_dividend(div_data))

        # [API 3] 최대주주
        sh_data = collector._request("hyslrSttus.json", params.copy())
        row_data.update(collector.process_major_shareholder(sh_data))

        # [API 4] 임직원
        emp_data = collector._request("empSttus.json", params.copy())
        row_data.update(collector.process_employees(emp_data))

        # [API 5] 회계감사
        audit_data = collector._request("accnutAuditOpn.json", params.copy())
        row_data.update(collector.process_auditor(audit_data))

        # [API 6~N] 미상환 잔액 및 기타 (단순 합계 또는 건수)
        # 회사채 미상환
        bond_data = collector._request("cprndNrdmpBlce.json", params.copy())
        row_data.update(collector.process_simple_sum(bond_data, 'rmndr_test_am', 'bond_unpaid_balance'))

        # 단기사채 미상환
        st_bond_data = collector._request("stbdNrdmpBlce.json", params.copy())
        row_data.update(collector.process_simple_sum(st_bond_data, 'rmndr_test_am', 'short_bond_unpaid_balance'))

        # 기업어음 미상환
        cp_data = collector._request("cpNrdmpBlce.json", params.copy())
        row_data.update(collector.process_simple_sum(cp_data, 'rmndr_test_am', 'cp_unpaid_balance'))

        # 자기주식 (취득/처분 내역 있으면 True/False 또는 건수)
        stock_data = collector._request("tesstkAcqsDspsSttus.json", params.copy())
        if stock_data:
            row_data['treasury_stock_event'] = 'Y'

        # 증자(감자) 현황 (건수만 체크)
        inc_data = collector._request("irdsSttus.json", params.copy())
        if inc_data:
            row_data['capital_change_count'] = len(inc_data)

        final_data_list.append(row_data)
        count += 1
        # time.sleep(0.1) # 전체 속도 조절

    # 4. CSV 저장
    if final_data_list:
        df = pd.DataFrame(final_data_list)

        # 종목코드 병합
        if not corp_df.empty:
            df = pd.merge(df, corp_df, on='corp_code', how='left')

        # 컬럼 순서 정리 (중요도 순)
        cols = ['corp_name', 'stock_code', 'bsns_year', 'report_name',
                'fin_revenue', 'fin_op_income', 'fin_net_income',
                'fin_total_assets', 'fin_total_equity', 'fin_total_liabilities',
                'div_dps_common',
                'emp_total_count', 'emp_avg_salary', 'auditor_name', 'audit_opinion',
                'bond_unpaid_balance', 'capital_change_count']

        # 존재하는 컬럼만 선택하여 정렬
        # 기존에 생성되어 있을 수 있는 컬럼을 안전하게 제거
        df = df.drop(columns=['major_shareholder_name', 'major_shareholder_ratio'], errors='ignore')
        final_cols = [c for c in cols if c in df.columns] + [c for c in df.columns if c not in cols]
        df = df[final_cols]

        save_path = "integrated_financial_data.csv"
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"\n✅ 저장 완료: {save_path}")
        display(df.head())
    else:
        print("❌ 수집된 데이터가 없습니다.")

if __name__ == "__main__":
    # 테스트를 위해 5개 기업만 수행 (전체 수행시 limit_companies=1000 등으로 변경)
    run_integrated_collection(limit_companies=2000)

✅ API Key Loaded: 1a2d...****
🔍 공시 목록 조회 중...
✅ 총 2792건의 정기보고서 발견.
[1] 신한알파리츠 (사업보고서 (2025.09)) 데이터 수집 중...
[2] KR모터스 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[3] 남해화학 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[4] 남해화학 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[5] 보락 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[6] 프레스티지바이오파마 (분기보고서 (2025.09)) 데이터 수집 중...
[7] 흥국화재 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[8] 한화 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[9] 롯데손해보험 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[10] 한화생명 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[11] 미래에셋생명 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[12] 메리츠금융지주 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[13] 엘브이엠씨 (분기보고서 (2025.09)) 데이터 수집 중...
[14] 동양생명 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[15] 코리안리 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[16] DSR제강 (분기보고서 (2025.09)) 데이터 수집 중...
[17] 한화손해보험 ([기재정정]분기보고서 (2025.09)) 데이터 수집 중...
[18] 솔루스첨단소재 ([기재정정]분기보고서 (2025.03)) 데이터 수집 중...
[19] 솔루스첨단소재 ([기재정정]사업보고서 (2024.12)) 데이터 수집 중...
[20] 산일전기 (분기보고서 (2025.09)) 데이터 수집 중...
[21] KC그린홀딩스 ([기재정정]분기보고서 (202

KeyboardInterrupt: 